In [7]:
from Bio import SeqIO
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import math

def find_box_motifs(fasta_file, max_mismatches=2):
    """Find Box A and Box B motifs"""
    
    degenerate_bases = {
        'R': 'AG', 'Y': 'CT', 'W': 'AT', 'N': 'ACGT',
        'A': 'A', 'C': 'C', 'G': 'G', 'T': 'T'
    }
    
    def matches_position(base, pattern_char):
        return base in degenerate_bases.get(pattern_char, '')
    
    def count_mismatches(sequence, pattern, start_pos):
        mismatches = 0
        match_details = []
        for i, pattern_char in enumerate(pattern):
            seq_base = sequence[start_pos + i]
            if not matches_position(seq_base, pattern_char):
                mismatches += 1
                match_details.append(f"pos {i}: {seq_base} ≠ {pattern_char}")
        return mismatches, match_details
    
    def find_best_matches(sequence, pattern):
        matches = []
        pattern_len = len(pattern)
        
        for i in range(len(sequence) - pattern_len + 1):
            mismatches, details = count_mismatches(sequence, pattern, i)
            if mismatches <= max_mismatches:
                matches.append({
                    'position': i,
                    'sequence': sequence[i:i + pattern_len],
                    'mismatches': mismatches,
                    'strand': '+',
                    'details': details
                })
        
        rev_comp = str(sequence.reverse_complement())
        for i in range(len(rev_comp) - pattern_len + 1):
            mismatches, details = count_mismatches(rev_comp, pattern, i)
            if mismatches <= max_mismatches:
                fwd_pos = len(sequence) - i - pattern_len
                matches.append({
                    'position': fwd_pos,
                    'sequence': rev_comp[i:i + pattern_len],
                    'mismatches': mismatches,
                    'strand': '-',
                    'details': details
                })
        
        matches.sort(key=lambda x: x['mismatches'])
        return matches
    
    box_a_pattern = "TGGCTCACGCC"
    box_b_pattern = "GTTCGAGAC"
    
    results = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq = record.seq.upper()
        box_a_matches = find_best_matches(seq, box_a_pattern)
        box_b_matches = find_best_matches(seq, box_b_pattern)
        
        results[record.id] = {
            'sequence': str(seq),
            'sequence_length': len(seq),
            'box_a': box_a_matches,
            'box_b': box_b_matches
        }
    
    return results


def visualize_sequences_wrapped(results_subset, output_file, chars_per_line=50):
    """
    Wrapped version with LARGE font (35pt), tight spacing, and Box A/B at same height
    Also marks position 99 in yellow
    """
    
    box_a_consensus = "TGGCTCACGCC"
    box_b_consensus = "GTTCGAGAC"
    
    n_sequences = len(results_subset)
    
    # Very large figure for large font
    fig = plt.figure(figsize=(30, 10*n_sequences))
    
    # LARGE monospace font
    mono_font = FontProperties(family='monospace', size=35)
    
    for seq_idx, (seq_id, data) in enumerate(results_subset.items()):
        sequence = data['sequence']
        seq_len = len(sequence)
        
        # Create color map - default black
        colors = ['black'] * seq_len
        
        # Mark position 99 (100th character, 0-indexed) in yellow
        if seq_len > 99:
            colors[99] = 'gold'  # Using 'gold' for better visibility than pure yellow
        
        best_box_a_pos = None
        best_box_a_mm = 0
        if data['box_a']:
            best_match = data['box_a'][0]
            best_box_a_pos = best_match['position']
            best_box_a_mm = best_match['mismatches']
            for i in range(best_match['position'], 
                          best_match['position'] + len(box_a_consensus)):
                if i < seq_len:
                    # Don't override position 99 marking
                    if i != 99:
                        colors[i] = 'red'
        
        best_box_b_pos = None
        best_box_b_mm = 0
        if data['box_b']:
            best_match = data['box_b'][0]
            best_box_b_pos = best_match['position']
            best_box_b_mm = best_match['mismatches']
            for i in range(best_match['position'], 
                          best_match['position'] + len(box_b_consensus)):
                if i < seq_len:
                    # Don't override position 99 marking
                    if i != 99:
                        colors[i] = 'blue'
        
        # Calculate number of lines
        n_lines = (seq_len + chars_per_line - 1) // chars_per_line
        
        ax = plt.subplot(n_sequences, 1, seq_idx + 1)
        ax.axis('off')
        
        # Title
        ax.text(0.01, 0.98, f"{seq_id} ({seq_len} bp)", 
               fontsize=28, weight='bold', transform=ax.transAxes)
        
        # Character width - VERY TIGHT, characters almost touching
        char_width = 0.0165  # Adjusted for font size 35 with minimal spacing
        x_seq_start = 0.08
        
        y_start = 0.92
        y_line_spacing = 0.10  # REDUCED vertical spacing
        
        for line_num in range(n_lines):
            start_idx = line_num * chars_per_line
            end_idx = min(start_idx + chars_per_line, seq_len)
            
            y_seq = y_start - (line_num * y_line_spacing)
            
            # Position number
            ax.text(0.01, y_seq, f"{start_idx:>5}:", 
                   fontsize=20, transform=ax.transAxes, family='monospace')
            
            # Draw sequence - characters very close together
            for i in range(start_idx, end_idx):
                x_pos = x_seq_start + (i - start_idx) * char_width
                ax.text(x_pos, y_seq, sequence[i], 
                       fontproperties=mono_font, color=colors[i], 
                       transform=ax.transAxes)
            
            # Consensus at same height for both boxes
            y_cons = y_seq - 0.035  # Tighter vertical spacing
            
            # Check if Box A match is on this line
            if best_box_a_pos is not None:
                if start_idx <= best_box_a_pos < end_idx:
                    for i, cons_char in enumerate(box_a_consensus):
                        cons_pos = best_box_a_pos + i
                        if start_idx <= cons_pos < end_idx:
                            x_pos = x_seq_start + (cons_pos - start_idx) * char_width
                            ax.text(x_pos, y_cons, cons_char, 
                                   fontproperties=mono_font, color='darkred',
                                   transform=ax.transAxes, weight='bold')
                    
                    # Label for Box A
                    x_label = x_seq_start + (best_box_a_pos - start_idx) * char_width
                    ax.text(x_label, y_cons - 0.025, 
                           f"Box A ({best_box_a_mm}mm)", 
                           fontsize=18, color='darkred', transform=ax.transAxes)
            
            # Check if Box B match is on this line (same height as Box A)
            if best_box_b_pos is not None:
                if start_idx <= best_box_b_pos < end_idx:
                    for i, cons_char in enumerate(box_b_consensus):
                        cons_pos = best_box_b_pos + i
                        if start_idx <= cons_pos < end_idx:
                            x_pos = x_seq_start + (cons_pos - start_idx) * char_width
                            ax.text(x_pos, y_cons, cons_char, 
                                   fontproperties=mono_font, color='darkblue',
                                   transform=ax.transAxes, weight='bold')
                    
                    # Label for Box B
                    x_label = x_seq_start + (best_box_b_pos - start_idx) * char_width
                    ax.text(x_label, y_cons - 0.025, 
                           f"Box B ({best_box_b_mm}mm)", 
                           fontsize=18, color='darkblue', transform=ax.transAxes)
        
        # Summary at bottom
        info_y = y_start - (n_lines * y_line_spacing) - 0.03
        
        summary_parts = []
        if data['box_a']:
            summary_parts.append(f"Box A: pos {best_box_a_pos} ({best_box_a_mm} mismatches)")
        if data['box_b']:
            summary_parts.append(f"Box B: pos {best_box_b_pos} ({best_box_b_mm} mismatches)")
        summary_parts.append("Position 99 marked in yellow")
        
        if summary_parts:
            ax.text(0.01, info_y, " | ".join(summary_parts), 
                   fontsize=18, transform=ax.transAxes, style='italic')
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"Visualization saved to: {output_file}")
    plt.close()


# Main execution
if __name__ == "__main__":
    fasta_file = "/scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/group1_0:11.fa"
    
    # Find motifs
    results = find_box_motifs(fasta_file, max_mismatches=2)
    
    # Create visualizations
    output_dir = "/scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/"
    
    # Split results into chunks of 5 sequences
    sequences_per_plot = 5
    seq_items = list(results.items())
    n_plots = math.ceil(len(seq_items) / sequences_per_plot)
    
    print(f"Creating {n_plots} plot(s) for {len(seq_items)} sequences...")
    
    for plot_num in range(n_plots):
        start_idx = plot_num * sequences_per_plot
        end_idx = min(start_idx + sequences_per_plot, len(seq_items))
        
        # Create subset dictionary for this plot
        subset = dict(seq_items[start_idx:end_idx])
        
        # Create filename with plot number
        if n_plots > 1:
            png_file = f"{output_dir}box_motifs_plot_{plot_num+1}_of_{n_plots}.png"
            svg_file = f"{output_dir}box_motifs_plot_{plot_num+1}_of_{n_plots}.svg"
        else:
            png_file = f"{output_dir}box_motifs.png"
            svg_file = f"{output_dir}box_motifs.svg"
        
        # Create visualization
        visualize_sequences_wrapped(subset, png_file, chars_per_line=50)
        visualize_sequences_wrapped(subset, svg_file, chars_per_line=50)
    
    print(f"\nAll visualizations saved to {output_dir}")

Creating 3 plot(s) for 11 sequences...
Visualization saved to: /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/box_motifs_plot_1_of_3.png
Visualization saved to: /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/box_motifs_plot_1_of_3.svg
Visualization saved to: /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/box_motifs_plot_2_of_3.png
Visualization saved to: /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/box_motifs_plot_2_of_3.svg
Visualization saved to: /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/box_motifs_plot_3_of_3.png
Visualization saved to: /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/box_motifs_plot_3_of_3.svg

All visualizations saved to /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/


In [8]:
from Bio import SeqIO
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
import math

def find_box_motifs(fasta_file, max_mismatches=2):
    """Find Box A and Box B motifs"""
    
    degenerate_bases = {
        'R': 'AG', 'Y': 'CT', 'W': 'AT', 'N': 'ACGT',
        'A': 'A', 'C': 'C', 'G': 'G', 'T': 'T'
    }
    
    def matches_position(base, pattern_char):
        return base in degenerate_bases.get(pattern_char, '')
    
    def count_mismatches_detailed(sequence, pattern, start_pos):
        """Return mismatches count and list of positions with mismatches"""
        mismatches = 0
        mismatch_positions = []
        for i, pattern_char in enumerate(pattern):
            seq_base = sequence[start_pos + i]
            if not matches_position(seq_base, pattern_char):
                mismatches += 1
                mismatch_positions.append(start_pos + i)
        return mismatches, mismatch_positions
    
    def find_best_matches(sequence, pattern):
        matches = []
        pattern_len = len(pattern)
        
        for i in range(len(sequence) - pattern_len + 1):
            mismatches, mismatch_pos = count_mismatches_detailed(sequence, pattern, i)
            if mismatches <= max_mismatches:
                matches.append({
                    'position': i,
                    'sequence': sequence[i:i + pattern_len],
                    'mismatches': mismatches,
                    'mismatch_positions': mismatch_pos,
                    'strand': '+',
                })
        
        rev_comp = str(sequence.reverse_complement())
        for i in range(len(rev_comp) - pattern_len + 1):
            mismatches, mismatch_pos = count_mismatches_detailed(rev_comp, pattern, i)
            if mismatches <= max_mismatches:
                fwd_pos = len(sequence) - i - pattern_len
                # Adjust mismatch positions to forward strand
                fwd_mismatch_pos = [len(sequence) - mp - 1 for mp in mismatch_pos]
                matches.append({
                    'position': fwd_pos,
                    'sequence': rev_comp[i:i + pattern_len],
                    'mismatches': mismatches,
                    'mismatch_positions': fwd_mismatch_pos,
                    'strand': '-',
                })
        
        matches.sort(key=lambda x: x['mismatches'])
        return matches
    
    box_a_pattern = "TGGCTCACGCC"
    box_b_pattern = "GTTCGAGAC"
    
    results = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq = record.seq.upper()
        box_a_matches = find_best_matches(seq, box_a_pattern)
        box_b_matches = find_best_matches(seq, box_b_pattern)
        
        results[record.id] = {
            'sequence': str(seq),
            'sequence_length': len(seq),
            'box_a': box_a_matches,
            'box_b': box_b_matches
        }
    
    return results


def create_heatmap_visualization(results_subset, output_file):
    """
    Create heatmap where:
    - Grey: regular bases
    - Dark red: Box A perfect match
    - Light red: Box A with mismatch
    - Dark blue: Box B perfect match
    - Light blue: Box B with mismatch
    - Yellow border: position 99
    """
    
    box_a_consensus = "TGGCTCACGCC"
    box_b_consensus = "GTTCGAGAC"
    
    # Define colors
    COLOR_GREY = 0.8          # Regular base
    COLOR_BOX_A_MATCH = 0.2   # Dark red (Box A perfect match)
    COLOR_BOX_A_MISMATCH = 0.5 # Light red (Box A mismatch)
    COLOR_BOX_B_MATCH = 0.0   # Dark blue (Box B perfect match)
    COLOR_BOX_B_MISMATCH = 0.4 # Light blue (Box B mismatch)
    COLOR_POS99 = 1.0         # Yellow (position 99)
    
    seq_ids = list(results_subset.keys())
    n_sequences = len(seq_ids)
    
    # Get max sequence length
    max_len = max(results_subset[sid]['sequence_length'] for sid in seq_ids)
    
    # Create matrices for red and blue channels
    red_channel = np.full((n_sequences, max_len), COLOR_GREY)
    blue_channel = np.full((n_sequences, max_len), COLOR_GREY)
    green_channel = np.full((n_sequences, max_len), COLOR_GREY)
    
    # Track position 99 for each sequence
    pos99_mask = np.zeros((n_sequences, max_len), dtype=bool)
    
    for seq_idx, seq_id in enumerate(seq_ids):
        data = results_subset[seq_id]
        seq_len = data['sequence_length']
        
        # Mark position 99 if sequence is long enough
        if seq_len > 99:
            pos99_mask[seq_idx, 99] = True
        
        # Process Box A matches (red)
        if data['box_a']:
            best_match = data['box_a'][0]
            start = best_match['position']
            end = start + len(box_a_consensus)
            mismatch_positions = set(best_match['mismatch_positions'])
            
            for i in range(start, min(end, seq_len)):
                if i in mismatch_positions:
                    # Light red for mismatch
                    red_channel[seq_idx, i] = 1.0
                    green_channel[seq_idx, i] = 0.6
                    blue_channel[seq_idx, i] = 0.6
                else:
                    # Dark red for perfect match
                    red_channel[seq_idx, i] = 0.8
                    green_channel[seq_idx, i] = 0.0
                    blue_channel[seq_idx, i] = 0.0
        
        # Process Box B matches (blue)
        if data['box_b']:
            best_match = data['box_b'][0]
            start = best_match['position']
            end = start + len(box_b_consensus)
            mismatch_positions = set(best_match['mismatch_positions'])
            
            for i in range(start, min(end, seq_len)):
                if i in mismatch_positions:
                    # Light blue for mismatch
                    red_channel[seq_idx, i] = 0.6
                    green_channel[seq_idx, i] = 0.6
                    blue_channel[seq_idx, i] = 1.0
                else:
                    # Dark blue for perfect match
                    red_channel[seq_idx, i] = 0.0
                    green_channel[seq_idx, i] = 0.0
                    blue_channel[seq_idx, i] = 0.9
        
        # Mark position 99 in yellow/gold
        if seq_len > 99:
            red_channel[seq_idx, 99] = 1.0
            green_channel[seq_idx, 99] = 0.84
            blue_channel[seq_idx, 99] = 0.0
    
    # Stack channels to create RGB image
    rgb_image = np.stack([red_channel, green_channel, blue_channel], axis=2)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(max(20, max_len * 0.15), n_sequences * 0.8))
    
    # Display heatmap
    ax.imshow(rgb_image, aspect='auto', interpolation='nearest')
    
    # Add gridlines for position 99
    for seq_idx in range(n_sequences):
        if pos99_mask[seq_idx, :].any():
            pos = 99
            # Draw a thick border around position 99
            rect = mpatches.Rectangle((pos - 0.5, seq_idx - 0.5), 1, 1,
                                     linewidth=3, edgecolor='black',
                                     facecolor='none')
            ax.add_patch(rect)
    
    # Set ticks
    ax.set_yticks(range(n_sequences))
    ax.set_yticklabels(seq_ids, fontsize=12)
    
    # X-axis ticks every 10 positions
    x_ticks = list(range(0, max_len, 10))
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_ticks, fontsize=10)
    
    # Add vertical line at position 99
    ax.axvline(x=99, color='black', linestyle='--', linewidth=2, alpha=0.5)
    
    ax.set_xlabel('Position', fontsize=14)
    ax.set_ylabel('Sequence', fontsize=14)
    ax.set_title('Box A and Box B Motif Matches', fontsize=16, weight='bold')
    
    # Create legend
    legend_elements = [
        mpatches.Patch(facecolor='grey', label='No match'),
        mpatches.Patch(facecolor=(0.8, 0.0, 0.0), label='Box A match'),
        mpatches.Patch(facecolor=(1.0, 0.6, 0.6), label='Box A mismatch'),
        mpatches.Patch(facecolor=(0.0, 0.0, 0.9), label='Box B match'),
        mpatches.Patch(facecolor=(0.6, 0.6, 1.0), label='Box B mismatch'),
        mpatches.Patch(facecolor='gold', edgecolor='black', linewidth=2, label='Position 99')
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1),
             fontsize=12, frameon=True)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"Heatmap saved to: {output_file}")
    plt.close()


# Main execution
if __name__ == "__main__":
    fasta_file = "/scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/group1_0:11.fa"
    
    # Find motifs
    results = find_box_motifs(fasta_file, max_mismatches=2)
    
    # Create visualizations
    output_dir = "/scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/"
    
    # Split results into chunks of appropriate size for heatmap
    sequences_per_plot = 20  # Can show more sequences in heatmap format
    seq_items = list(results.items())
    n_plots = math.ceil(len(seq_items) / sequences_per_plot)
    
    print(f"Creating {n_plots} heatmap(s) for {len(seq_items)} sequences...")
    
    for plot_num in range(n_plots):
        start_idx = plot_num * sequences_per_plot
        end_idx = min(start_idx + sequences_per_plot, len(seq_items))
        
        # Create subset dictionary for this plot
        subset = dict(seq_items[start_idx:end_idx])
        
        # Create filename with plot number
        if n_plots > 1:
            png_file = f"{output_dir}box_motifs_heatmap_{plot_num+1}_of_{n_plots}.png"
            svg_file = f"{output_dir}box_motifs_heatmap_{plot_num+1}_of_{n_plots}.svg"
        else:
            png_file = f"{output_dir}box_motifs_heatmap.png"
            svg_file = f"{output_dir}box_motifs_heatmap.svg"
        
        # Create heatmap
        create_heatmap_visualization(subset, png_file)
        create_heatmap_visualization(subset, svg_file)
    
    print(f"\nAll heatmaps saved to {output_dir}")

Creating 1 heatmap(s) for 11 sequences...
Heatmap saved to: /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/box_motifs_heatmap.png
Heatmap saved to: /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/box_motifs_heatmap.svg

All heatmaps saved to /scratch/st-cdeboer-1/sambina/position_mpra/outputs/7-repeat_elements/
